# Phase 0 — Data Orientation & Diagnostic

This notebook reproduces the full Phase 0 workflow:
1. **Diagnostic** — column-by-column quality audit of the raw dataset
2. **Cleaning** — applying all documented cleaning decisions to produce the clean dataset

> **Before running:** activate the lab virtualenv (`source venv/bin/activate`) and install dependencies (`pip install -r requirements.txt`).

All outputs (figures, CSVs) are saved to `phase0_diagnostic/`. Run cells top to bottom.


## Part 1 — Data Quality Diagnostic

Loads the raw CSV and produces a full quality audit: missing values, impossible values, encoding inconsistencies, duplicates. Saves 13 diagnostic charts and a `quality_summary.csv`.

In [1]:
"""
Phase 0 — Data Diagnostic Script
===================================
Produces a full quality audit of the raw student_wellness.csv dataset.

Outputs:
  - phase0_diagnostic/figures/  : one chart per column with quality issues
  - phase0_diagnostic/quality_summary.csv : issue log
  - Prints a full per-column quality report to stdout
"""

import sys
import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE   = "/Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab"
RAW    = os.path.join(BASE, "dataset", "student_wellness.csv")
FIGS   = os.path.join(BASE, "phase0_diagnostic", "figures")
os.makedirs(FIGS, exist_ok=True)

# ── Palette ───────────────────────────────────────────────────────────────────
PASTEL = ["#A8C8E8", "#F4A8B0", "#A8E8C8", "#F4D8A8", "#C8A8E8", "#F4F4A8"]
WARN   = "#F4A8B0"   # rose for bad values
OK     = "#A8C8E8"   # blue for good values

LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#4A4A4A"),
    plot_bgcolor="#FAFAFA",
    paper_bgcolor="#FFFFFF",
    margin=dict(t=80, b=60, l=60, r=40),
)

def save_fig(fig, name):
    path = os.path.join(FIGS, name)
    fig.write_image(path, width=1200, height=600, scale=2)
    print(f"  [saved] {name}")

# ── Load raw data ──────────────────────────────────────────────────────────────
df = pd.read_csv(RAW, dtype=str)  # load everything as string first
print(f"\n{'='*60}")
print(f"RAW DATASET — {RAW}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"{'='*60}\n")

issues = []  # will collect: {column, issue_type, affected_rows, recommendation}

# ══════════════════════════════════════════════════════════════════════════════
# 1. DUPLICATE ROWS
# ══════════════════════════════════════════════════════════════════════════════
dupes = df.duplicated()
n_dupes = dupes.sum()
print(f"[DUPLICATES] {n_dupes} duplicate rows found")
if n_dupes > 0:
    issues.append({"column": "ALL", "issue_type": "Duplicate rows",
                   "affected_rows": int(n_dupes),
                   "recommendation": "Drop duplicates using df.drop_duplicates()"})

# ── Duplicate bar chart ────────────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=["Unique rows", "Duplicate rows"],
    y=[len(df) - n_dupes, n_dupes],
    marker_color=[OK, WARN],
    text=[len(df) - n_dupes, n_dupes],
    textposition="auto",
))
fig.update_layout(**LAYOUT,
    title="Duplicate Row Check",
    yaxis_title="Count",
)
save_fig(fig, "00_duplicates.png")

# ══════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES
# ══════════════════════════════════════════════════════════════════════════════
missing = df.replace("", np.nan).isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_pct", ascending=False)

print(f"\n[MISSING VALUES] Columns with missing data:")
print(missing_df.to_string())

for col in missing_df.index:
    issues.append({"column": col, "issue_type": "Missing values",
                   "affected_rows": int(missing_df.loc[col, "missing_count"]),
                   "recommendation": f"Impute with median (numeric) or mode (categorical), or flag as unknown"})

# ── Missing values bar chart ───────────────────────────────────────────────────
if not missing_df.empty:
    fig = px.bar(missing_df.reset_index(), x="index", y="missing_pct",
                 color="missing_pct", color_continuous_scale=["#A8C8E8", "#F4A8B0"],
                 text="missing_pct", labels={"index": "Column", "missing_pct": "% Missing"})
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.update_layout(**LAYOUT, title="Missing Values by Column (%)",
                      coloraxis_showscale=False)
    save_fig(fig, "01_missing_values.png")

# ══════════════════════════════════════════════════════════════════════════════
# 3. COLUMN-BY-COLUMN QUALITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════

# ── age ────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[age]")
age_num = pd.to_numeric(df["age"], errors="coerce")
invalid_age = age_num[(age_num < 16) | (age_num > 35)].dropna()
print(f"  Valid range: 16–35 | Invalid values: {len(invalid_age)}")
print(f"  Min: {age_num.min()}, Max: {age_num.max()}, Unique invalid: {sorted(invalid_age.unique())}")
if len(invalid_age):
    issues.append({"column": "age", "issue_type": "Impossible values (out of range)",
                   "affected_rows": len(invalid_age),
                   "recommendation": "Set to NaN and impute with median age (≈21)"})

fig = go.Figure()
fig.add_trace(go.Histogram(x=age_num[age_num.between(16, 35)], name="Valid", marker_color=OK,
                           nbinsx=20, opacity=0.8))
fig.add_trace(go.Histogram(x=age_num[~age_num.between(16, 35)], name="Invalid (<16 or >35)",
                           marker_color=WARN, nbinsx=20, opacity=0.9))
fig.update_layout(**LAYOUT, title="Age Distribution — Valid vs Invalid Values",
                  xaxis_title="Age", yaxis_title="Count", barmode="overlay",
                  legend=dict(x=0.75, y=0.9))
save_fig(fig, "02_age_check.png")

# ── gpa ────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[gpa]")
gpa_num = pd.to_numeric(df["gpa"], errors="coerce")
invalid_gpa = gpa_num[(gpa_num < 0) | (gpa_num > 4.0)].dropna()
missing_gpa = gpa_num.isna().sum()
print(f"  Valid range: 0–4.0 | Out-of-range: {len(invalid_gpa)} | Missing: {missing_gpa}")
print(f"  Out-of-range values: {sorted(invalid_gpa.round(2).unique())}")
if len(invalid_gpa):
    issues.append({"column": "gpa", "issue_type": "Out-of-range values (GPA > 4.0 or < 0)",
                   "affected_rows": len(invalid_gpa),
                   "recommendation": "Cap at 4.0 for values slightly above; set to NaN for clearly wrong values (GPA > 4.5)"})

valid_gpa = gpa_num[gpa_num.between(0, 4.0)]
fig = make_subplots(rows=1, cols=2, subplot_titles=["GPA Distribution (valid only)", "GPA Range Check"])
fig.add_trace(go.Histogram(x=valid_gpa, marker_color=OK, nbinsx=25, name="Valid GPA"), row=1, col=1)
fig.add_trace(go.Box(y=gpa_num, marker_color=WARN, name="All GPA (incl. outliers)",
                     boxpoints="outliers", jitter=0.3), row=1, col=2)
fig.update_layout(**LAYOUT, title="GPA — Distribution & Outlier Check", showlegend=False)
save_fig(fig, "03_gpa_check.png")

# ── study_hours_per_day ────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[study_hours_per_day]")
study_num = pd.to_numeric(df["study_hours_per_day"], errors="coerce")
invalid_study = study_num[(study_num < 0) | (study_num > 18)].dropna()
print(f"  Valid range: 0–18 | Invalid: {len(invalid_study)}")
print(f"  Values: {sorted(invalid_study.round(1).unique())}")
if len(invalid_study):
    issues.append({"column": "study_hours_per_day", "issue_type": "Impossible values (>18 or <0)",
                   "affected_rows": len(invalid_study),
                   "recommendation": "Set to NaN — 30hrs/day is physically impossible"})

fig = go.Figure()
fig.add_trace(go.Box(y=study_num, marker_color=PASTEL[0],
                     boxpoints="outliers", jitter=0.3, name="Study Hours"))
fig.add_hline(y=18, line_dash="dash", line_color=WARN,
              annotation_text="Max reasonable (18hrs)", annotation_position="right")
fig.add_hline(y=0, line_dash="dash", line_color=WARN,
              annotation_text="Min (0hrs)", annotation_position="right")
fig.update_layout(**LAYOUT, title="Study Hours per Day — Box Plot with Outlier Threshold",
                  yaxis_title="Hours")
save_fig(fig, "04_study_hours_check.png")

# ── sleep_hours_per_night ──────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[sleep_hours_per_night]")
sleep_num = pd.to_numeric(df["sleep_hours_per_night"], errors="coerce")
invalid_sleep = sleep_num[(sleep_num < 0) | (sleep_num > 16)].dropna()
print(f"  Valid range: 0–16 | Invalid: {len(invalid_sleep)}")
print(f"  Values: {sorted(invalid_sleep.round(1).unique())}")
if len(invalid_sleep):
    issues.append({"column": "sleep_hours_per_night", "issue_type": "Impossible values (negative or >16)",
                   "affected_rows": len(invalid_sleep),
                   "recommendation": "Set to NaN — negative sleep is impossible; >16 hrs is implausible for a student"})

fig = go.Figure()
fig.add_trace(go.Box(y=sleep_num, marker_color=PASTEL[2], boxpoints="outliers", jitter=0.3))
fig.add_hline(y=0, line_dash="dash", line_color=WARN, annotation_text="Floor (0)")
fig.add_hline(y=16, line_dash="dash", line_color=WARN, annotation_text="Ceiling (16hrs)")
fig.update_layout(**LAYOUT, title="Sleep Hours per Night — Box Plot with Range Check",
                  yaxis_title="Hours")
save_fig(fig, "05_sleep_hours_check.png")

# ── attendance_rate ────────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[attendance_rate]")
att_num = pd.to_numeric(df["attendance_rate"], errors="coerce")
invalid_att = att_num[att_num > 100].dropna()
print(f"  Valid range: 0–100 | Values > 100: {len(invalid_att)}")
print(f"  Example > 100 values: {sorted(invalid_att.unique())[:8]}")
if len(invalid_att):
    issues.append({"column": "attendance_rate", "issue_type": "Values > 100% (impossible percentage)",
                   "affected_rows": len(invalid_att),
                   "recommendation": "Cap at 100% — a percentage cannot exceed 100"})

fig = go.Figure()
fig.add_trace(go.Histogram(x=att_num[att_num <= 100], name="Valid (≤100%)", marker_color=OK, opacity=0.8))
fig.add_trace(go.Histogram(x=att_num[att_num > 100], name="Invalid (>100%)", marker_color=WARN, opacity=0.9))
fig.add_vline(x=100, line_dash="dash", line_color="#888",
              annotation_text="100% ceiling", annotation_position="top left")
fig.update_layout(**LAYOUT, title="Attendance Rate — Valid vs. Invalid Values",
                  xaxis_title="Attendance (%)", yaxis_title="Count", barmode="overlay")
save_fig(fig, "06_attendance_check.png")

# ── gender ─────────────────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[gender]")
gender_counts = df["gender"].value_counts()
print(f"  Unique values ({len(gender_counts)}):")
print(gender_counts.to_string())
inconsistent_gender = df["gender"].isin(["M", "male", "MALE", "man", "F", "female", "FEMALE", "woman"])
n_inconsistent = inconsistent_gender.sum()
print(f"  Inconsistent encodings: {n_inconsistent} rows")
if n_inconsistent:
    issues.append({"column": "gender", "issue_type": "Inconsistent categorical encoding",
                   "affected_rows": int(n_inconsistent),
                   "recommendation": "Standardize: map M/male/MALE→Male, F/female/FEMALE→Female"})

fig = px.bar(gender_counts.reset_index(), x="gender", y="count",
             color="gender", color_discrete_sequence=PASTEL,
             text="count", labels={"gender": "Gender Value", "count": "Count"})
fig.update_traces(textposition="outside")
fig.update_layout(**LAYOUT, title="Gender Column — All Unique Values (showing encoding inconsistencies)",
                  showlegend=False)
save_fig(fig, "07_gender_encoding.png")

# ── stress_level ───────────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[stress_level]")
stress_text = df["stress_level"][pd.to_numeric(df["stress_level"], errors="coerce").isna()]
n_text_stress = len(stress_text)
stress_text_vals = stress_text.value_counts()
print(f"  Non-numeric stress_level entries: {n_text_stress}")
print(stress_text_vals.to_string())
if n_text_stress:
    issues.append({"column": "stress_level", "issue_type": "Mixed types (numeric 1–10 + text labels)",
                   "affected_rows": int(n_text_stress),
                   "recommendation": "Map text labels to numeric: low→2.5, medium→5, high→7.5, very high→9.5; then convert to float"})

type_counts = pd.Series({"Numeric (1–10)": len(df) - n_text_stress, "Text label": n_text_stress})
fig = px.pie(values=type_counts.values, names=type_counts.index,
             color_discrete_sequence=[OK, WARN])
fig.update_layout(**LAYOUT, title="stress_level — Numeric vs. Text Label Breakdown")
save_fig(fig, "08_stress_type_mix.png")

# ── on_campus ──────────────────────────────────────────────────────────────────
print(f"\n{'─'*40}\n[on_campus]")
on_campus_vals = df["on_campus"].value_counts()
print(f"  Unique values ({len(on_campus_vals)}): {list(on_campus_vals.index)}")
inconsistent_oc = df["on_campus"].isin(["1", "0", "yes", "no", "YES", "NO", "true", "false"])
n_incon_oc = inconsistent_oc.sum()
print(f"  Non-boolean encodings: {n_incon_oc} rows")
if n_incon_oc:
    issues.append({"column": "on_campus", "issue_type": "Mixed boolean representations (True/False/1/0/yes/no)",
                   "affected_rows": int(n_incon_oc),
                   "recommendation": "Map all truthy variants → True, falsy variants → False; store as boolean"})

fig = px.bar(on_campus_vals.reset_index(), x="on_campus", y="count",
             color="on_campus", color_discrete_sequence=PASTEL,
             text="count", labels={"on_campus": "on_campus value", "count": "Count"})
fig.update_traces(textposition="outside")
fig.update_layout(**LAYOUT, title="on_campus — All Unique Values (showing boolean inconsistencies)",
                  showlegend=False)
save_fig(fig, "09_on_campus_encoding.png")

# ── anxiety_score & depression_score ──────────────────────────────────────────
for col, max_val, scale_name in [("anxiety_score", 21, "GAD-7"), ("depression_score", 27, "PHQ-9")]:
    print(f"\n{'─'*40}\n[{col}]")
    num = pd.to_numeric(df[col], errors="coerce")
    missing_n = num.isna().sum()
    invalid_n = num[(num < 0) | (num > max_val)].dropna()
    print(f"  {scale_name} scale (0–{max_val}) | Missing: {missing_n} | Out-of-range: {len(invalid_n)}")

# ── Summary chart: missing % heatmap per column ────────────────────────────────
all_missing_pct = {}
for col in df.columns:
    num_missing = df[col].replace("", np.nan).isnull().sum()
    all_missing_pct[col] = round(num_missing / len(df) * 100, 1)

missing_series = pd.Series(all_missing_pct)
fig = px.bar(x=missing_series.index, y=missing_series.values,
             color=missing_series.values,
             color_continuous_scale=["#A8E8C8", "#A8C8E8", "#F4D8A8", "#F4A8B0"],
             labels={"x": "Column", "y": "% Missing"},
             text=[f"{v}%" for v in missing_series.values])
fig.update_traces(textposition="outside")
fig.update_layout(**LAYOUT, title="Missing Values (%) — All Columns Overview",
                  coloraxis_showscale=False, xaxis_tickangle=-35)
save_fig(fig, "10_missing_all_columns.png")

# ══════════════════════════════════════════════════════════════════════════════
# 4. QUALITY SUMMARY CSV
# ══════════════════════════════════════════════════════════════════════════════
issues_df = pd.DataFrame(issues)
summary_path = os.path.join(BASE, "phase0_diagnostic", "quality_summary.csv")
issues_df.to_csv(summary_path, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# 5. FINAL PRINT SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("QUALITY AUDIT COMPLETE")
print(f"{'='*60}")
print(f"Total issues found: {len(issues)}")
print(f"\nIssue breakdown:")
for _, row in issues_df.iterrows():
    print(f"  [{row['column']}] {row['issue_type']} — {row['affected_rows']} rows")
    print(f"    → {row['recommendation']}")
print(f"\nFigures saved to: {FIGS}")
print(f"Quality summary saved to: {summary_path}")
print(f"\nNext step: Review each issue above, decide on cleaning strategy, then run cleaning.py")



RAW DATASET — /Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab/dataset/student_wellness.csv
Shape: 535 rows × 21 columns

[DUPLICATES] 3 duplicate rows found


  [saved] 00_duplicates.png

[MISSING VALUES] Columns with missing data:
                        missing_count  missing_pct
anxiety_score                      69         12.9
depression_score                   69         12.9
sleep_hours_per_night              53          9.9
monthly_spending                   48          9.0
gpa                                42          7.9
caffeine_mg_per_day                37          6.9
exercise_days_per_week             32          6.0
has_part_time_job                  26          4.9
num_clubs                          21          3.9


  [saved] 01_missing_values.png

────────────────────────────────────────
[age]
  Valid range: 16–35 | Invalid values: 6
  Min: -3.0, Max: 999.0, Unique invalid: [np.float64(-3.0), np.float64(0.0), np.float64(5.0), np.float64(150.0), np.float64(200.0), np.float64(999.0)]


  [saved] 02_age_check.png

────────────────────────────────────────
[gpa]
  Valid range: 0–4.0 | Out-of-range: 8 | Missing: 42
  Out-of-range values: [np.float64(-1.0), np.float64(-0.5), np.float64(4.75), np.float64(4.8), np.float64(4.99), np.float64(5.2), np.float64(5.5), np.float64(6.0)]


  [saved] 03_gpa_check.png

────────────────────────────────────────
[study_hours_per_day]
  Valid range: 0–18 | Invalid: 5
  Values: [np.float64(-2.0), np.float64(25.0), np.float64(28.0), np.float64(30.0), np.float64(35.0)]


...

  [saved] 06_attendance_check.png

────────────────────────────────────────
[gender]
  Unique values (12):
gender
Female               206
Male                 201
Non-binary            34
Prefer not to say     19
M                     12
F                     12
man                   11
FEMALE                10
male                  10
female                 7
MALE                   7
woman                  6
  Inconsistent encodings: 75 rows


  [saved] 07_gender_encoding.png

────────────────────────────────────────
[stress_level]
  Non-numeric stress_level entries: 20
stress_level
low          8
medium       6
high         4
very high    2


  [saved] 08_stress_type_mix.png

────────────────────────────────────────
[on_campus]
  Unique values (10): ['True', 'False', 'YES', 'true', '1', '0', 'false', 'yes', 'NO', 'no']
  Non-boolean encodings: 60 rows


  [saved] 09_on_campus_encoding.png

────────────────────────────────────────
[anxiety_score]
  GAD-7 scale (0–21) | Missing: 69 | Out-of-range: 0

────────────────────────────────────────
[depression_score]
  PHQ-9 scale (0–27) | Missing: 69 | Out-of-range: 0


  [saved] 10_missing_all_columns.png

QUALITY AUDIT COMPLETE
Total issues found: 18

Issue breakdown:
  [ALL] Duplicate rows — 3 rows
    → Drop duplicates using df.drop_duplicates()
  [anxiety_score] Missing values — 69 rows
    → Impute with median (numeric) or mode (categorical), or flag as unknown
  [depression_score] Missing values — 69 rows
    → Impute with median (numeric) or mode (categorical), or flag as unknown
  [sleep_hours_per_night] Missing values — 53 rows
    → Impute with median (numeric) or mode (categorical), or flag as unknown
  [monthly_spending] Missing values — 48 rows
    → Impute with median (numeric) or mode (categorical), or flag as unknown
  [gpa] Missing values — 42 rows
    → Impute with median (numeric) or mode (categorical), or flag as unknown
  [caffeine_mg_per_day] Missing values — 37 rows
    → Impute with median (numeric) or mode (categorical), or flag as unknown
  [exercise_days_per_week] Missing values — 32 rows
    → Impute with median (numeric) 

## Part 2 — Data Cleaning

Applies all cleaning decisions documented in `phase0_diagnostic/report.md`:
- Drop 3 duplicate rows
- Null and impute impossible values (age, GPA, study hours, sleep hours)
- Cap attendance > 100% at 100%
- Standardize gender encoding (12 variants → 4 canonical)
- Map stress_level text labels to numeric (low→2.5, medium→5.0, high→7.5, very high→9.5)
- Standardize on_campus boolean (10 variants → True/False)
- Median imputation for all remaining missing numeric values

Produces `dataset/student_wellness_clean.csv` used in all subsequent phases.
Saves 3 before/after comparison charts.


In [2]:
"""
Phase 0 — Data Cleaning Script
================================
Applies all cleaning decisions documented in phase0_diagnostic/report.md.
Produces the cleaned dataset used in all subsequent phases.

Outputs:
  - dataset/student_wellness_clean.csv
  - phase0_diagnostic/figures/before_after_*.png  (before/after comparisons)
"""

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE  = "/Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab"
RAW   = os.path.join(BASE, "dataset", "student_wellness.csv")
CLEAN = os.path.join(BASE, "dataset", "student_wellness_clean.csv")
FIGS  = os.path.join(BASE, "phase0_diagnostic", "figures")

PASTEL = ["#A8C8E8", "#F4A8B0", "#A8E8C8", "#F4D8A8", "#C8A8E8", "#F4F4A8"]
LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#4A4A4A"),
    plot_bgcolor="#FAFAFA", paper_bgcolor="#FFFFFF",
    margin=dict(t=80, b=60, l=60, r=40),
)

def save_fig(fig, name):
    fig.write_image(os.path.join(FIGS, name), width=1200, height=600, scale=2)
    print(f"  [saved] {name}")

# ── Load raw ───────────────────────────────────────────────────────────────────
df = pd.read_csv(RAW, dtype=str)
n_raw = len(df)
print(f"\nRaw shape: {df.shape}")

cleaning_log = []  # {step, column, action, rows_affected}

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Drop duplicate rows
# ══════════════════════════════════════════════════════════════════════════════
before = len(df)
df = df.drop_duplicates()
dropped_dupes = before - len(df)
cleaning_log.append({"step": 1, "column": "ALL", "action": "Dropped duplicate rows",
                      "rows_affected": dropped_dupes})
print(f"\n[1] Dropped {dropped_dupes} duplicate rows → {len(df)} rows remain")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Convert numeric columns, null out impossible values
# ══════════════════════════════════════════════════════════════════════════════

# age: valid 16–35
df["age"] = pd.to_numeric(df["age"], errors="coerce")
invalid_age_mask = ~df["age"].between(16, 35)
n_invalid_age = invalid_age_mask.sum()
df.loc[invalid_age_mask, "age"] = np.nan
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median).round(0).astype(int)
cleaning_log.append({"step": 2, "column": "age",
                      "action": f"Nulled {n_invalid_age} impossible values; imputed with median ({age_median:.0f})",
                      "rows_affected": int(n_invalid_age)})
print(f"[2] age: nulled {n_invalid_age} impossible values, imputed median={age_median:.0f}")

# gpa: valid 0–4.0
df["gpa"] = pd.to_numeric(df["gpa"], errors="coerce")
over_gpa = df["gpa"] > 4.0
df.loc[over_gpa & (df["gpa"] <= 4.3), "gpa"] = 4.0      # slight rounding → cap
df.loc[over_gpa & (df["gpa"] > 4.3), "gpa"] = np.nan    # clearly wrong → NaN
df.loc[df["gpa"] < 0, "gpa"] = np.nan
n_invalid_gpa = df["gpa"].isna().sum()
gpa_median = df["gpa"].median()
df["gpa"] = df["gpa"].fillna(gpa_median).round(2)
cleaning_log.append({"step": 3, "column": "gpa",
                      "action": f"Capped GPA≤4.3→4.0, nulled clearly wrong, imputed median={gpa_median:.2f}",
                      "rows_affected": int(over_gpa.sum())})
print(f"[3] gpa: fixed out-of-range values, imputed median={gpa_median:.2f}")

# study_hours_per_day: valid 0–18
df["study_hours_per_day"] = pd.to_numeric(df["study_hours_per_day"], errors="coerce")
invalid_study = ~df["study_hours_per_day"].between(0, 18)
n_inv_study = invalid_study.sum()
df.loc[invalid_study, "study_hours_per_day"] = np.nan
study_median = df["study_hours_per_day"].median()
df["study_hours_per_day"] = df["study_hours_per_day"].fillna(study_median).round(1)
cleaning_log.append({"step": 4, "column": "study_hours_per_day",
                      "action": f"Nulled {n_inv_study} impossible values; imputed median ({study_median:.1f})",
                      "rows_affected": int(n_inv_study)})
print(f"[4] study_hours_per_day: nulled {n_inv_study}, imputed median={study_median:.1f}")

# sleep_hours_per_night: valid 2–16
df["sleep_hours_per_night"] = pd.to_numeric(df["sleep_hours_per_night"], errors="coerce")
invalid_sleep = ~df["sleep_hours_per_night"].between(2, 16)
n_inv_sleep = invalid_sleep.sum()
df.loc[invalid_sleep, "sleep_hours_per_night"] = np.nan
sleep_median = df["sleep_hours_per_night"].median()
df["sleep_hours_per_night"] = df["sleep_hours_per_night"].fillna(sleep_median).round(1)
cleaning_log.append({"step": 5, "column": "sleep_hours_per_night",
                      "action": f"Nulled {n_inv_sleep} impossible values; imputed median ({sleep_median:.1f})",
                      "rows_affected": int(n_inv_sleep)})
print(f"[5] sleep_hours_per_night: nulled {n_inv_sleep}, imputed median={sleep_median:.1f}")

# attendance_rate: cap at 100
df["attendance_rate"] = pd.to_numeric(df["attendance_rate"], errors="coerce")
over_att = df["attendance_rate"] > 100
n_over_att = over_att.sum()
df.loc[over_att, "attendance_rate"] = 100.0
cleaning_log.append({"step": 6, "column": "attendance_rate",
                      "action": f"Capped {n_over_att} values >100% to 100%",
                      "rows_affected": int(n_over_att)})
print(f"[6] attendance_rate: capped {n_over_att} values >100%")

# screen_time_hours, social_media_hours, life_satisfaction — convert + impute
for col in ["screen_time_hours", "social_media_hours", "life_satisfaction"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    n_null = df[col].isna().sum()
    if n_null > 0:
        df[col] = df[col].fillna(df[col].median())

# caffeine, exercise, monthly_spending, anxiety, depression, num_clubs — convert + impute
for col in ["caffeine_mg_per_day", "exercise_days_per_week", "monthly_spending",
            "anxiety_score", "depression_score", "num_clubs"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    med = df[col].median()
    n_null = df[col].isna().sum()
    df[col] = df[col].fillna(med).round(1)
    if n_null > 0:
        cleaning_log.append({"step": 7, "column": col,
                              "action": f"Imputed {n_null} missing values with median ({med:.1f})",
                              "rows_affected": int(n_null)})
        print(f"[7] {col}: imputed {n_null} missing with median={med:.1f}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Standardize gender encoding
# ══════════════════════════════════════════════════════════════════════════════
gender_map = {
    "M": "Male", "male": "Male", "MALE": "Male", "man": "Male",
    "F": "Female", "female": "Female", "FEMALE": "Female", "woman": "Female",
}
n_gender_fix = df["gender"].isin(gender_map.keys()).sum()
df["gender"] = df["gender"].replace(gender_map)
cleaning_log.append({"step": 8, "column": "gender",
                      "action": f"Standardized {n_gender_fix} inconsistent encodings",
                      "rows_affected": int(n_gender_fix)})
print(f"\n[8] gender: standardized {n_gender_fix} entries")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Fix stress_level mixed types
# ══════════════════════════════════════════════════════════════════════════════
stress_text_map = {"low": 2.5, "medium": 5.0, "high": 7.5, "very high": 9.5}
is_text_stress = df["stress_level"].isin(stress_text_map.keys())
n_text_stress = is_text_stress.sum()
df["stress_level"] = df["stress_level"].replace(stress_text_map)
df["stress_level"] = pd.to_numeric(df["stress_level"], errors="coerce")
stress_median = df["stress_level"].median()
df["stress_level"] = df["stress_level"].fillna(stress_median).round(1)
cleaning_log.append({"step": 9, "column": "stress_level",
                      "action": f"Mapped {n_text_stress} text labels to numeric (low→2.5, medium→5, high→7.5, very high→9.5)",
                      "rows_affected": int(n_text_stress)})
print(f"[9] stress_level: converted {n_text_stress} text labels to numeric")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Fix on_campus mixed booleans
# ══════════════════════════════════════════════════════════════════════════════
true_vals  = {"True", "true", "1", "yes", "YES"}
false_vals = {"False", "false", "0", "no", "NO"}
n_oc_before = (~df["on_campus"].isin(["True", "False"])).sum()
df["on_campus"] = df["on_campus"].apply(
    lambda x: True if x in true_vals else (False if x in false_vals else np.nan)
)
df["on_campus"] = df["on_campus"].fillna(True)  # impute with mode
cleaning_log.append({"step": 10, "column": "on_campus",
                      "action": f"Standardized {n_oc_before} mixed boolean values",
                      "rows_affected": int(n_oc_before)})
print(f"[10] on_campus: standardized {n_oc_before} mixed boolean values")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — has_part_time_job — impute missing with mode
# ══════════════════════════════════════════════════════════════════════════════
n_ptj_missing = df["has_part_time_job"].isna().sum()
if n_ptj_missing == 0:
    n_ptj_missing = (df["has_part_time_job"] == "nan").sum()
df["has_part_time_job"] = df["has_part_time_job"].replace("nan", np.nan)
mode_ptj = df["has_part_time_job"].mode()[0]
df["has_part_time_job"] = df["has_part_time_job"].fillna(mode_ptj)
print(f"[11] has_part_time_job: imputed {n_ptj_missing} missing with mode='{mode_ptj}'")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — year_in_school — convert to int
# ══════════════════════════════════════════════════════════════════════════════
df["year_in_school"] = pd.to_numeric(df["year_in_school"], errors="coerce")
df["year_in_school"] = df["year_in_school"].fillna(df["year_in_school"].median()).round(0).astype(int)

# ══════════════════════════════════════════════════════════════════════════════
# BEFORE/AFTER COMPARISON CHARTS
# ══════════════════════════════════════════════════════════════════════════════
df_raw = pd.read_csv(RAW, dtype=str)

# GPA before / after
gpa_before = pd.to_numeric(df_raw["gpa"], errors="coerce").dropna()
gpa_after  = df["gpa"]
fig = make_subplots(rows=1, cols=2, subplot_titles=["GPA — Before Cleaning", "GPA — After Cleaning"])
fig.add_trace(go.Histogram(x=gpa_before, nbinsx=30, marker_color="#F4A8B0", name="Before"), row=1, col=1)
fig.add_trace(go.Histogram(x=gpa_after,  nbinsx=30, marker_color="#A8C8E8", name="After"),  row=1, col=2)
fig.update_layout(**LAYOUT, title="GPA Distribution: Before vs After Cleaning", showlegend=False)
save_fig(fig, "11_gpa_before_after.png")

# Sleep before / after
sleep_before = pd.to_numeric(df_raw["sleep_hours_per_night"], errors="coerce").dropna()
sleep_after  = df["sleep_hours_per_night"]
fig = make_subplots(rows=1, cols=2, subplot_titles=["Sleep — Before Cleaning", "Sleep — After Cleaning"])
fig.add_trace(go.Histogram(x=sleep_before, nbinsx=25, marker_color="#F4A8B0", name="Before"), row=1, col=1)
fig.add_trace(go.Histogram(x=sleep_after,  nbinsx=25, marker_color="#A8C8E8", name="After"),  row=1, col=2)
fig.update_layout(**LAYOUT, title="Sleep Hours: Before vs After Cleaning", showlegend=False)
save_fig(fig, "12_sleep_before_after.png")

# Gender encoding before / after
gender_before = df_raw["gender"].value_counts()
gender_after  = df["gender"].value_counts()
fig = make_subplots(rows=1, cols=2, subplot_titles=["Gender — Before (12 variants)", "Gender — After (4 canonical)"])
fig.add_trace(go.Bar(x=gender_before.index, y=gender_before.values, marker_color="#F4A8B0"), row=1, col=1)
fig.add_trace(go.Bar(x=gender_after.index,  y=gender_after.values,  marker_color="#A8C8E8"), row=1, col=2)
fig.update_layout(**LAYOUT, title="Gender Encoding: Before vs After Cleaning", showlegend=False)
save_fig(fig, "13_gender_before_after.png")

# ══════════════════════════════════════════════════════════════════════════════
# SAVE CLEAN DATASET
# ══════════════════════════════════════════════════════════════════════════════
df.to_csv(CLEAN, index=False)

# Cleaning log
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(os.path.join(BASE, "phase0_diagnostic", "cleaning_log.csv"), index=False)

print(f"\n{'='*60}")
print("CLEANING COMPLETE")
print(f"{'='*60}")
print(f"Raw rows:     {n_raw}")
print(f"Clean rows:   {len(df)}")
print(f"Rows removed: {n_raw - len(df)} (duplicates)")
print(f"\nClean dataset: {CLEAN}")
print(f"Cleaning log:  {os.path.join(BASE, 'phase0_diagnostic', 'cleaning_log.csv')}")
print(f"\nFinal dtypes:")
print(df.dtypes.to_string())



Raw shape: (535, 21)

[1] Dropped 3 duplicate rows → 532 rows remain
[2] age: nulled 6 impossible values, imputed median=21
[3] gpa: fixed out-of-range values, imputed median=3.12
[4] study_hours_per_day: nulled 5, imputed median=7.0
[5] sleep_hours_per_night: nulled 58, imputed median=6.8
[6] attendance_rate: capped 10 values >100%
[7] caffeine_mg_per_day: imputed 37 missing with median=189.0
[7] exercise_days_per_week: imputed 32 missing with median=3.0
[7] monthly_spending: imputed 48 missing with median=814.6
[7] anxiety_score: imputed 69 missing with median=6.0
[7] depression_score: imputed 69 missing with median=5.0
[7] num_clubs: imputed 21 missing with median=3.0

[8] gender: standardized 75 entries
[9] stress_level: converted 20 text labels to numeric
[10] on_campus: standardized 60 mixed boolean values
[11] has_part_time_job: imputed 26 missing with mode='No'


  [saved] 11_gpa_before_after.png


  [saved] 12_sleep_before_after.png


  [saved] 13_gender_before_after.png

CLEANING COMPLETE
Raw rows:     535
Clean rows:   532
Rows removed: 3 (duplicates)

Clean dataset: /Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab/dataset/student_wellness_clean.csv
Cleaning log:  /Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab/phase0_diagnostic/cleaning_log.csv

Final dtypes:
student_id                    str
age                         int64
gender                        str
major                         str
year_in_school              int64
gpa                       float64
study_hours_per_day       float64
attendance_rate           float64
sleep_hours_per_night     float64
exercise_days_per_week    float64
screen_time_hours         float64
social_media_hours        float64
caffeine_mg_per_day       float64
stress_level              float64
anxiety_score             float64
depression_score          float64
life_satisfaction         float64
num_clubs                 float64
on_campus                  

## Summary

After running both cells above:
- **Raw → Clean:** 535 rows → 532 rows (3 duplicates removed)
- **Issues resolved:** 18 distinct data quality issues across 10 columns
- **Clean dataset:** `dataset/student_wellness_clean.csv`
- **Key caveats:** anxiety/depression missing values may reflect response bias; stress_level text imputation is approximate

See `phase0_diagnostic/report.md` for the full column-by-column findings and cleaning decision log.
